# Mini Project 3: Iris Flower Clustering Project

**Dataset:** Iris Dataset (UCI / Kaggle / scikit-learn)  
**Topic:** Unsupervised Machine Learning, K-Means Clustering, and PCA Dimensionality Reduction  

---

### Project Objectives:
1. **Load and Explore** the Iris dataset.
2. **Feature Standardization**: Prepare features using `StandardScaler`.
3. **Elbow Method & Silhouette Analysis**: Discover the optimal number of clusters $k$.
4. **PCA Dimensionality Reduction**: Reduce 4-dimensional features to 2 principal components ($>95.8\%$ variance retained).
5. **K-Means Clustering ($k=3$)**: Fit K-Means on the standardized data.
6. **Scatter Plot Visualizations**: Plot clusters and centroids in 2D PCA space and feature space.
7. **Cluster Comparison**: Match unsupervised clusters to true species and evaluate performance via Confusion Matrix, Adjusted Rand Index (ARI), and V-Measure.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    confusion_matrix,
    classification_report
)
from scipy.optimize import linear_sum_assignment

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
%matplotlib inline


## Step 1: Load and Explore the Iris Dataset
The dataset contains 150 samples across 3 species (Setosa, Versicolor, Virginica), with 4 measurements:
- Sepal Length (cm)
- Sepal Width (cm)
- Petal Length (cm)
- Petal Width (cm)


In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))

print(f"Dataset Shape: {df.shape}")
print(df.head())
print("\nSummary Statistics:")
print(df.describe().round(2))


## Step 2: Feature Standardization
K-Means computes Euclidean distance $d(p, q) = \sqrt{\sum (p_i - q_i)^2}$.
Standardization with `StandardScaler` ($\mu = 0, \sigma = 1$) ensures features with larger numerical ranges don't dominate the distance metric.


In [ ]:
X = df[iris.feature_names].values
y_true = df['target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Feature Means:", X_scaled.mean(axis=0).round(4))
print("Scaled Feature Standard Deviations:", X_scaled.std(axis=0).round(4))


## Step 3: Finding Optimal $k$ (Elbow Method & Silhouette Scores)
We test $k \in [1, 10]$:
- **Inertia (WCSS)**: Measures cluster compactness (within-cluster sum of squared distances to centroid).
- **Silhouette Score**: Measures how similar an object is to its own cluster compared to neighboring clusters.


In [ ]:
k_range = range(1, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    if k >= 2:
        silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Method Plot
ax1.plot(list(k_range), inertias, marker='o', color='#1f77b4', linewidth=2.5)
ax1.axvline(3, color='#d62728', linestyle='--', label='Elbow Inflection (k=3)')
ax1.set_title("Elbow Method (Inertia vs Number of Clusters)")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia (WCSS)")
ax1.set_xticks(list(k_range))
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# Silhouette Scores Plot
ax2.plot(range(2, 11), silhouette_scores, marker='s', color='#2ca02c', linewidth=2.5)
ax2.axvline(3, color='#d62728', linestyle='--', label='Target k=3')
ax2.set_title("Silhouette Scores across k")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Coefficient")
ax2.set_xticks(range(2, 11))
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## Step 4: Dimensionality Reduction via PCA
Principal Component Analysis orthogonalizes the 4-dimensional features into principal components along the axes of maximal variance.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
total_var = np.sum(pca.explained_variance_ratio_) * 100

print(f"PC1 Explained Variance: {pc1_var:.2f}%")
print(f"PC2 Explained Variance: {pc2_var:.2f}%")
print(f"Cumulative 2D Explained Variance: {total_var:.2f}%")

# Loadings
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=iris.feature_names)
print("\nPCA Component Loadings:")
print(loadings.round(3))


## Step 5: Apply K-Means Clustering ($k=3$)
We fit K-Means with $k=3$ on the standardized data and calculate cluster centroids.


In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=20, max_iter=300, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# Project centroids into 2D PCA coordinate space
pca_centroids = np.zeros((3, 2))
for c in range(3):
    pca_centroids[c] = X_pca[cluster_labels == c].mean(axis=0)

print(f"K-Means converged in {kmeans.n_iter_} iterations.")
print(f"Final Inertia: {kmeans.inertia_:.2f}")
print(f"Silhouette Score (k=3): {silhouette_score(X_scaled, cluster_labels):.4f}")


## Step 6: Visualizing Clusters (Scatter Plots)
We compare:
1. **Ground Truth** (True Species) in 2D PCA Space.
2. **K-Means Predicted Clusters** with Centroids in 2D PCA Space.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# 1. True Species
palette_species = {'setosa': '#1f77b4', 'versicolor': '#ff7f0e', 'virginica': '#2ca02c'}
for idx, name in enumerate(iris.target_names):
    mask = (y_true == idx)
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=name.capitalize(), color=palette_species[name], s=65, edgecolors='k', alpha=0.85)

axes[0].set_title("Ground Truth: Iris Species (2D PCA Space)")
axes[0].set_xlabel(f"PC1 ({pc1_var:.1f}% Variance)")
axes[0].set_ylabel(f"PC2 ({pc2_var:.1f}% Variance)")
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle=':', alpha=0.6)

# 2. Predicted Clusters
cluster_colors = ['#d62728', '#9467bd', '#17becf']
for c in range(3):
    mask = (cluster_labels == c)
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=f"Cluster {c}", color=cluster_colors[c], s=65, edgecolors='k', alpha=0.85)

# Centroids
axes[1].scatter(pca_centroids[:, 0], pca_centroids[:, 1], s=250, c='gold', marker='X', edgecolors='k', linewidths=1.8, label="Centroids", zorder=10)
for c_id, (cx, cy) in enumerate(pca_centroids):
    axes[1].annotate(f" C{c_id}", (cx, cy), fontsize=11, fontweight='bold', xytext=(6, 6), textcoords='offset points')

axes[1].set_title("Unsupervised: K-Means (k=3) Clusters & Centroids")
axes[1].set_xlabel(f"PC1 ({pc1_var:.1f}% Variance)")
axes[1].legend(frameon=True)
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.suptitle("PCA 2D Projection: Ground Truth vs K-Means Clusters", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 7: Compare Predicted Clusters vs True Labels
Because K-Means is unsupervised, cluster indexes {0, 1, 2} do not have predefined names.
We construct the contingency matrix and use the **Hungarian Algorithm** (`scipy.optimize.linear_sum_assignment`) to find the optimal assignment between clusters and species.


In [ ]:
contingency = np.zeros((3, 3), dtype=int)
for true, pred in zip(y_true, cluster_labels):
    contingency[true, pred] += 1

row_ind, col_ind = linear_sum_assignment(-contingency)
mapping = {col: row for row, col in zip(row_ind, col_ind)}
mapped_pred = np.array([mapping[c] for c in cluster_labels])

print("Optimal Cluster Assignment Mapping:")
for c, r in mapping.items():
    print(f"  Cluster {c} -> {iris.target_names[r].capitalize()} (Class {r})")

# Accuracy and Metrics
acc = np.mean(mapped_pred == y_true)
ari = adjusted_rand_score(y_true, cluster_labels)
ami = adjusted_mutual_info_score(y_true, cluster_labels)

print(f"\nClustering Accuracy (Aligned): {acc * 100:.2f}%")
print(f"Adjusted Rand Index (ARI):     {ari:.4f}")
print(f"Adjusted Mutual Info (AMI):    {ami:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, mapped_pred, target_names=iris.target_names))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, mapped_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[n.capitalize() for n in iris.target_names],
            yticklabels=[n.capitalize() for n in iris.target_names],
            annot_kws={'size': 13, 'weight': 'bold'})
ax.set_title("Confusion Matrix: Aligned K-Means vs True Species")
ax.set_xlabel("Predicted Species")
ax.set_ylabel("True Species")
plt.tight_layout()
plt.show()


## Step 8: Key Insights and Discussion
1. **Setosa is 100% Separable**:
   - Iris-setosa forms an isolated cluster in feature space and PCA space, achieving precision=1.00 and recall=1.00.
2. **Versicolor and Virginica Boundary**:
   - Iris-versicolor and Iris-virginica have overlapping petal and sepal distributions.
   - K-Means assigns points strictly based on spherical Euclidean boundaries, resulting in misclassifications near the border, reaching an overall accuracy of **83.33%** and ARI of **0.6201**.
3. **PCA Effectiveness**:
   - PC1 and PC2 retain **95.81%** of variance, proving that 2 dimensions are sufficient to visualize and capture almost all geometric structure of the 4D flower measurements.
